# Agentic Data Agent — Client DB Mount

Mounts all `client/db/db-1..db-16` databases with their documentation and BIRD-style question–SQL pairs.

**Docker PostgreSQL:** Containers on ports 5436–5451. Run `./scripts/docker_postgres_qa.sh -a` to load schema+data if needed.

## 1. Setup

In [ ]:
%pip install -q psycopg2-binary

import sys
from pathlib import Path

ROOT = Path.cwd()
if "doc" in str(ROOT) and (ROOT / ".." / "..").resolve().exists():
    ROOT = (ROOT / ".." / "..").resolve()  # client/doc -> repo root
elif "client" not in str(ROOT):
    ROOT = (ROOT / "..").resolve() if (ROOT / ".." / "client").exists() else ROOT
ROOT = ROOT.resolve()
CLIENT = ROOT / "client"
CLIENT_DB = CLIENT / "db"
CLIENT_DOC = CLIENT / "doc"
SCRIPTS = ROOT / "scripts"
sys.path.insert(0, str(SCRIPTS))

from agentic_mount import load_client_db, get_bird_pairs, get_pg_port

DB_NUMS = list(range(1, 17))

print(f"ROOT: {ROOT}")
print(f"CLIENT: {CLIENT}")
print(f"Ports: {get_pg_port(1)}–{get_pg_port(16)}")

## 2. Docker Check

In [ ]:
import subprocess

def check_docker() -> bool:
    try:
        r = subprocess.run(["docker", "info"], capture_output=True, timeout=5)
        return r.returncode == 0
    except Exception:
        return False

def check_containers() -> dict:
    try:
        r = subprocess.run(
            ["docker", "ps", "--format", "{{.Names}}"],
            capture_output=True, text=True, timeout=5
        )
        names = [n.strip() for n in r.stdout.splitlines() if "postgres-db-" in n]
        return {n: True for n in names}
    except Exception:
        return {}

docker_ok = check_docker()
containers = check_containers() if docker_ok else {}
print(f"Docker: {'✓' if docker_ok else '✗'}")
print(f"PostgreSQL containers: {len(containers)}/16")

## 3. Mount Client Databases

In [ ]:
mounted = {n: load_client_db(CLIENT_DB, n) for n in DB_NUMS}

for n, m in mounted.items():
    qty = len(m.get("queries", []))
    docs_ok = "✓" if m.get("docs") else "✗"
    print(f"  {m['db']}: {qty} queries, docs={docs_ok}")

## 4. Run SQL and BIRD Pairs

In [ ]:
import psycopg2

def get_connection(db_num: int):
    return psycopg2.connect(
        host="localhost", port=get_pg_port(db_num),
        user="postgres", password="postgres", dbname=f"db{db_num}",
        connect_timeout=3
    )

def run_sql(db_num: int, sql: str, limit: int = 100) -> tuple:
    try:
        conn = get_connection(db_num)
        cur = conn.cursor()
        q = sql.strip()
        cur.execute(q)
        cols = [d[0] for d in cur.description] if cur.description else []
        rows = cur.fetchall()
        conn.close()
        return (rows, cols)
    except Exception as e:
        return (None, str(e))

# BIRD-style pairs for db-2
pairs = get_bird_pairs(mounted[2], 2)
for p in pairs[:3]:
    print(f"Q{p['number']}: {str(p['question'])[:60]}...")

## 5. Example: Run Gold SQL

In [ ]:
db_num = 2
pair = get_bird_pairs(mounted[db_num], db_num)[0]
rows, cols = run_sql(db_num, pair["sql"])

if rows is not None:
    print(f"Rows: {len(rows)}, Cols: {cols}")
    try:
        import pandas as pd
        df = pd.DataFrame(rows, columns=cols)
        display(df.head())
    except Exception:
        print(rows[:5])
else:
    print(f"Error: {cols}")